# NumPy ile küçük GRU — hafızalı konuşma iyileştirme deneyi

Ömer Altın ve Muhammed Aslan — EE491–EE492

Bu model **GRU (Gated Recurrent Unit)** kullanır. GRU bir recurrent neural network
birimidir; **spiking neural network değildir**. İleri yayılım, zaman boyunca geri
yayılım ve Adam güncellemesi NumPy ile yazılmıştır. SciPy yalnız ses/STFT işlemlerindedir.

Mimari: **257 girdi → 48 birimli GRU → 257 sigmoid maske**; 56.657 parametre.
Her çerçevede gürültülü log-genlik spektrumu ve önceki gizli durum kullanılır.
Beş çerçeveyi yan yana koyan MLP'den farklı olarak bir gizli durum zaman boyunca taşınır.

Başlangıç MLP'siyle aynı küçük DNS alt kümesi ve kaynaklara göre ayrılmış veri grupları
kullanılır. NumPy MLP, kısa bağlamlı MLP ve GRU ayrı deneylerdir. Bu GRU, projenin
nihai Spiking-FullSubNet uygulaması değildir.

**Hızlı deneme:** defteri Colab'a yükleyin, hücreleri sırayla çalıştırın. Veri isterse
eski DNS_MLP_Baslangic.zip dosyasını; ağırlık isterse bu paketteki gru_weights.npz dosyasını
seçin. Varsayılan ayarda tekrar eğitim beklenmez. Eğitim hücresindeki
TRAIN_FROM_SCRATCH=True ile ağı sıfırdan eğitebilirsiniz.

In [ ]:
from pathlib import Path
import csv, json, time, io, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import ShortTimeFFT

ROOT = Path('/content/DNS_MLP_Baslangic')
if (Path.cwd() / 'pairs.csv').exists():
    ROOT = Path.cwd()
if not (ROOT / 'pairs.csv').exists():
    from google.colab import files
    print('Dosyaları Seç düğmesinden önceki DNS_MLP_Baslangic.zip VERİ dosyasını seçin.')
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        assert 'DNS_MLP_Baslangic/pairs.csv' in archive.namelist(), 'Önceki veri ZIP dosyasını seçin.'
        archive.extractall('/content')
    ROOT = Path('/content/DNS_MLP_Baslangic')
    del uploaded

OUTPUT_DIR = ROOT / 'numpy_gru_v4'
OUTPUT_DIR.mkdir(exist_ok=True)
with (ROOT / 'pairs.csv').open(encoding='utf-8', newline='') as f:
    pairs = list(csv.DictReader(f))
transform = ShortTimeFFT(np.hanning(512), hop=128, fs=16000, fft_mode='onesided')
HIDDEN = 48
EPOCHS = 15
BATCH = 16
TBPTT = 64
LEARNING_RATE = 0.001
print('Veri hazır:', ROOT)
print('Girdi 257 → GRU 48 → sigmoid maske 257; 56.657 parametre')

## 1. Sesler ve dizi biçimindeki veriler

`read_audio` 16 kHz mono PCM16 kaydı okur. `prepare` girdiyi
**(kayıt sayısı, zaman çerçevesi, frekans)** olarak hazırlar. Train (420,628,257),
validation (90,628,257) olur. Zaman çerçeveleri karıştırılmaz; yalnız tam kayıtların
eğitimdeki sırası karıştırılır. Böylece geçmiş bilgisi doğru sırada işlenir.

Girdi log1p(|Y|), hedef Re(S·conj(Y))/max(|Y|²,1e−8), ağırlık P=|Y|²'dir.
S temiz eğitim referansı, Y gürültülü STFT'dir. Bu hedef önceki maske MLP'siyle aynıdır.
Çıkış maskesi sigmoid nedeniyle 0–1 arasındadır; hedef T bu aralık dışında olabilir.

In [ ]:
def read_audio(relative):
    fs,x=wavfile.read(ROOT/relative)
    assert fs==16000 and x.shape==(80000,) and x.dtype==np.int16
    return x.astype(np.float64)/32768


def prepare(split):
    rows=[r for r in pairs if r['split']==split]
    frames=transform.stft(read_audio(rows[0]['noisy_path'])).shape[1]
    X=np.empty((len(rows),frames,257),dtype=np.float32)
    T=np.empty_like(X);P=np.empty_like(X)
    for i,row in enumerate(rows):
        clean=transform.stft(read_audio(row['clean_path']))
        noisy=transform.stft(read_audio(row['noisy_path']))
        power=abs(noisy)**2
        X[i]=np.log1p(abs(noisy)).T
        T[i]=(np.real(clean*np.conj(noisy))/np.maximum(power,1e-8)).T
        P[i]=power.T
    print('Prepared',split,X.shape,flush=True)
    return X,T,P

## 2. GRU'nun ileri yayılımı

Her çerçevede x mevcut girdi, h önceki gizli durumdur. Kod satır-vektör kullanır:

$$z_t=\sigma(x_tW_z+h_{t-1}U_z+b_z),\quad
  r_t=\sigma(x_tW_r+h_{t-1}U_r+b_r).$$
$$q_t=\tanh(x_tW_q+(r_t\odot h_{t-1})U_q+b_q).$$
$$h_t=(1-z_t)\odot h_{t-1}+z_t\odot q_t,\qquad
  M_t=\sigma(h_tW_o+b_o).$$

r sıfırlama kapısıdır; aday durumun geçmişten ne ölçüde yararlanacağını belirler.
Bu kodda z **yeni adayın ağırlığıdır**: z=0 eski durumu korur, z=1 yeni adayı kullanır.
Bazı kaynaklar z ile 1−z'nin yerini değiştirir; geri yayılım da burada yazılan denkleme göredir.
Bu hafıza geçmişin kusursuz kaydı değil, öğrenilmiş sınırlı boyutlu bir özetidir.

`forward_gru` aynı kaydın sonraki parçasına h0 ile devam edebilir. Yeni bağımsız
kayıtta h0 verilmez ve gizli durum sıfırdan başlar. Başlangıç çıkış maskesi 0.9'dur.

In [ ]:
def sigmoid(x):
    return np.exp(-np.logaddexp(0,-x))


def initialize():
    rng=np.random.default_rng(42)
    p={}
    for gate in ['z','r','q']:
        p['W'+gate]=(rng.standard_normal((257,HIDDEN))/np.sqrt(257)).astype(np.float32)
        q,_=np.linalg.qr(rng.standard_normal((HIDDEN,HIDDEN)))
        p['U'+gate]=(q*.9).astype(np.float32)
        p['b'+gate]=np.zeros((1,HIDDEN),dtype=np.float32)
    p['Wo']=np.zeros((HIDDEN,257),dtype=np.float32)
    p['bo']=np.full((1,257),np.log(.9/.1),dtype=np.float32)
    return p


def forward_gru(X,p,h0=None,keep_cache=True):
    # X: (record batch, time, frequency). h0 carries previous CHUNK state.
    h=np.zeros((len(X),p['Uz'].shape[0]),dtype=X.dtype) if h0 is None else h0.copy()
    masks=np.empty_like(X)
    caches=[] if keep_cache else None
    for t in range(X.shape[1]):
        x=X[:,t,:];previous=h
        z=sigmoid(x@p['Wz']+previous@p['Uz']+p['bz'])
        r=sigmoid(x@p['Wr']+previous@p['Ur']+p['br'])
        q=np.tanh(x@p['Wq']+(r*previous)@p['Uq']+p['bq'])
        # Our z weights the CANDIDATE. Other conventions interchange z and 1-z.
        h=(1-z)*previous+z*q
        masks[:,t,:]=sigmoid(h@p['Wo']+p['bo'])
        if keep_cache:caches.append((x,previous,z,r,q,h))
    return masks,h,caches


def objective(mask,target,power):
    return float(np.sum(power*(mask-target)**2,dtype=np.float64)/(np.sum(power,dtype=np.float64)+1e-12))

## 3. Zaman boyunca geri yayılım ve Adam

`backward_gru` çerçeveleri sondan başa dolaşır. Bir gizli duruma hem mevcut çıkışın
hatası hem de sonraki çerçeveden gelen türev katkıda bulunur. Kapıların ve aday
durumun türevleri W/U/b parametrelerine aktarılır.

`adam_step` gradyanın küresel normunu en fazla 5 yapar, ardından Adam uygular:
learning rate 0.001, β1=0.9, β2=0.999. Momentum ve ikinci moment için başlangıç
yanlılığı düzeltmesi kullanılır. Hazır bir eğitim kütüphanesi çağrılmaz.

In [ ]:
def backward_gru(mask,target,power,caches,p):
    grads={k:np.zeros_like(v) for k,v in p.items()}
    dm=2*power*(mask-target)/(np.sum(power,dtype=np.float64)+1e-12)
    dh_next=np.zeros_like(caches[0][1])
    for t in reversed(range(mask.shape[1])):
        x,previous,z,r,q,h=caches[t]
        do=dm[:,t,:]*mask[:,t,:]*(1-mask[:,t,:])
        grads['Wo']+=h.T@do;grads['bo']+=do.sum(axis=0,keepdims=True)
        dh=do@p['Wo'].T+dh_next
        dq=dh*z;dz=dh*(q-previous);dh_previous=dh*(1-z)
        aq=dq*(1-q*q)
        grads['Wq']+=x.T@aq;grads['Uq']+=(r*previous).T@aq
        grads['bq']+=aq.sum(axis=0,keepdims=True)
        dproduct=aq@p['Uq'].T
        dr=dproduct*previous;dh_previous+=dproduct*r
        ar=dr*r*(1-r)
        grads['Wr']+=x.T@ar;grads['Ur']+=previous.T@ar
        grads['br']+=ar.sum(axis=0,keepdims=True)
        dh_previous+=ar@p['Ur'].T
        az=dz*z*(1-z)
        grads['Wz']+=x.T@az;grads['Uz']+=previous.T@az
        grads['bz']+=az.sum(axis=0,keepdims=True)
        dh_previous+=az@p['Uz'].T
        dh_next=dh_previous
    return grads


def adam_step(p,grads,moments,velocities,step):
    # Global norm clipping is applied BEFORE the Adam update.
    norm=np.sqrt(sum(float(np.sum(g.astype(np.float64)**2)) for g in grads.values()))
    scale=min(1.,5./max(norm,1e-12))
    for key in p:
        g=grads[key]*scale
        moments[key]*=.9;moments[key]+=.1*g
        velocities[key]*=.999;velocities[key]+=.001*g*g
        corrected_m=moments[key]/(1-.9**step)
        corrected_v=velocities[key]/(1-.999**step)
        p[key]-=LEARNING_RATE*corrected_m/(np.sqrt(corrected_v)+1e-8)

## 4. Doğrulama ve eğitim düzeni

15 epoch, aynı anda 16 kayıt, 64 çerçevelik parçalarda truncated BPTT kullanılır.
64 adım × 8 ms = 512 ms'de bir gradyan zinciri kesilir; **gizli durumun sayısal değeri
aynı kaydın sonraki parçasına taşınır**. Her yeni kayıt grubunda sıfırlanır. Eğitimde
uzun bağımlılıkları doğrudan öğrenme, bu kesilmiş gradyanla sınırlıdır.

Normalizasyon yalnız train grubundan öğrenilir. En yüksek ortalama **doğrulama
dalga biçimi SNR'si** veren epoch kaydedilir; test grubuyla seçim yapılmaz. SNR
dalga biçimi hatasını ölçer, algısal kalitenin tek başına kanıtı değildir.

Önceki MLP'de SGD ve doğrulama maske kaybıyla seçim vardı. Bu deneyde mimari,
optimizer ve model seçim ölçütü birlikte değişti. Sonuç farkını sadece hafızanın
etkisini izole eden bir karşılaştırma olarak sunmayın.

Eğitim grafiğindeki online train loss, epoch boyunca değişen ağırlıklarla hesaplanan
kayıptır; validation loss ise epoch sonunda sabit ağırlıklarla hesaplanır.

In [ ]:
def predict_sequences(X,p):
    # Each sample is an entire separate file; h is zeroed for every batch.
    predictions=np.empty_like(X)
    for start in range(0,len(X),BATCH):
        predictions[start:start+BATCH]=forward_gru(X[start:start+BATCH],p,keep_cache=False)[0]
    return predictions


def snr(reference,estimate):
    return float(10*np.log10((np.sum(reference**2)+1e-12)/(np.sum((reference-estimate)**2)+1e-12)))


def validation_audio_scores(masks):
    rows=[r for r in pairs if r['split']=='validation']
    values=[]
    for mask,row in zip(masks,rows):
        clean=read_audio(row['clean_path']);noisy=read_audio(row['noisy_path'])
        output=transform.istft(mask.T*transform.stft(noisy),k1=len(noisy))
        values.append(snr(clean,output))
    return float(np.mean(values))


def run_training():
    X,T,P=prepare('train');V,U,Q=prepare('validation')
    # Accumulate over records, avoiding full float64 copies of large arrays.
    mean=np.mean(X,axis=(0,1),keepdims=True,dtype=np.float64)
    variance=np.zeros_like(mean)
    for record in X:
        d=record.astype(np.float64)-mean[0]
        variance+=np.sum(d*d,axis=0,keepdims=True)[None,:,:]
    std=np.maximum(np.sqrt(variance/(X.shape[0]*X.shape[1])),1e-4).astype(np.float32)
    mean=mean.astype(np.float32)
    X-=mean;X/=std;V-=mean;V/=std
    p=initialize()
    moments={k:np.zeros_like(v) for k,v in p.items()};velocities={k:np.zeros_like(v) for k,v in p.items()}
    best={k:v.copy() for k,v in p.items()}
    initial=predict_sequences(V,p)
    best_snr=validation_audio_scores(initial);best_loss=objective(initial,U,Q);best_epoch=0
    rng=np.random.default_rng(123);history=[];step=0
    for epoch in range(1,EPOCHS+1):
        tic=time.perf_counter();order=rng.permutation(len(X));numerator=denominator=0.
        for start in range(0,len(X),BATCH):
            ii=order[start:start+BATCH]
            # Slots keep the same recording for all chunks in this batch.
            h=None
            for t in range(0,X.shape[1],TBPTT):
                xb=X[ii,t:t+TBPTT,:];tb=T[ii,t:t+TBPTT,:];pb=P[ii,t:t+TBPTT,:]
                masks,h,caches=forward_gru(xb,p,h0=h)
                numerator+=float(np.sum(pb*(masks-tb)**2,dtype=np.float64))
                denominator+=float(np.sum(pb,dtype=np.float64))
                grads=backward_gru(masks,tb,pb,caches,p)
                step+=1;adam_step(p,grads,moments,velocities,step)
                # h is carried numerically; derivatives stop at each chunk edge.
        predicted=predict_sequences(V,p)
        val=objective(predicted,U,Q);val_snr=validation_audio_scores(predicted)
        history.append({'epoch':epoch,'online_train_loss':numerator/denominator,
                        'validation_loss':val,'validation_snr':val_snr,
                        'seconds':time.perf_counter()-tic})
        if val_snr>best_snr:
            best_snr=val_snr;best_loss=val;best_epoch=epoch;best={k:v.copy() for k,v in p.items()}
        print('EPOCH',epoch,'online_train',round(numerator/denominator,6),'val_loss',round(val,6),
              'val_snr',round(val_snr,4),'seconds',round(time.perf_counter()-tic,2),flush=True)
    np.savez(OUTPUT_DIR/'gru_weights.npz',**best,x_mean=mean,x_std=std,
        hidden=np.array(HIDDEN),fs=np.array(16000),window=np.array(512),hop=np.array(128),
        best_epoch=np.array(best_epoch),best_validation_snr=np.array(best_snr),
        best_validation_loss=np.array(best_loss),tbptt=np.array(TBPTT))
    (OUTPUT_DIR/'gru_training_history.json').write_text(json.dumps(history,indent=2))
    print('BEST_EPOCH',best_epoch,'VALIDATION_SNR',best_snr,flush=True)
    return best,mean,std

## 5. Hazır modeli yükleme veya sıfırdan eğitim

`TRAIN_FROM_SCRATCH=False` hazır modeli yükler; gru_weights.npz dosyasını seçin.
True yaparsanız bir önceki hücrede tanımlanan eğitimi başlatır. Her epoch'un süresi
ekranda yazılır. NumPy CPU kullanır; Colab'da GPU seçmek bu kodu GPU'ya taşımaz.

Hazır ağırlıkları yüklemek hızlıdır; veri yükleme, çıkarım ve ölçüm yine yapılır.
Defteri Drive'a kaydedin; runtime dosyaları ve Python değişkenleri kalıcı değildir.

In [ ]:
# False: hazır ağırlıkları yükle. True: NumPy ile sıfırdan eğit.
TRAIN_FROM_SCRATCH = False

if TRAIN_FROM_SCRATCH:
    gru_params, gru_mean, gru_std = run_training()
else:
    checkpoint_path = OUTPUT_DIR / 'gru_weights.npz'
    if not checkpoint_path.exists():
        from google.colab import files
        print('Bu güncelleme paketindeki gru_weights.npz dosyasını seçin.')
        uploaded_weights = files.upload()
        assert 'gru_weights.npz' in uploaded_weights, 'gru_weights.npz dosyasını seçin.'
        checkpoint_path.write_bytes(uploaded_weights['gru_weights.npz'])
        del uploaded_weights
    with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
        assert int(checkpoint['hidden']) == HIDDEN
        keys = ['Wz','Uz','bz','Wr','Ur','br','Wq','Uq','bq','Wo','bo']
        gru_params = {k: checkpoint[k].copy() for k in keys}
        gru_mean = checkpoint['x_mean'].copy()
        gru_std = checkpoint['x_std'].copy()
        print('Seçilen epoch:', int(checkpoint['best_epoch']))
        print('Seçimde kullanılan doğrulama SNR:', float(checkpoint['best_validation_snr']))

assert gru_params['Wz'].shape == (257, HIDDEN)
assert gru_mean.shape == gru_std.shape == (1, 1, 257)
print('Model hazır.')

history_path = OUTPUT_DIR / 'gru_training_history.json'
if history_path.exists():
    history = json.loads(history_path.read_text())
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
    epochs = [h['epoch'] for h in history]
    axes[0].plot(epochs, [h['online_train_loss'] for h in history], label='Online training')
    axes[0].plot(epochs, [h['validation_loss'] for h in history], label='Validation')
    axes[0].set(xlabel='Epoch', ylabel='Power-weighted mask MSE')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs, [h['validation_snr'] for h in history])
    axes[1].set(xlabel='Epoch', ylabel='Mean validation waveform SNR (dB)')
    axes[1].grid(alpha=0.3)
    fig.savefig(OUTPUT_DIR / 'gru_training_curve.png', dpi=150)
    plt.show()

## 6. Ses üretme ve tüm grupları ölçme

`enhance` sadece gürültülü sesi alır. 0–1 maskeyi gürültülü kompleks STFT ile çarpar
ve ters STFT uygular. Temiz referans yalnız ölçümde kullanılır; çıkarıma verilmez.
Gürültülü faz korunur ve ağ genliği artıramaz; bu iki kısıt da mükemmel ses garantisini engeller.

Gelecek STFT çerçeveleri GRU'ya verilmez. Kullanılan merkezli 32 ms STFT penceresi
kendi pencere gecikmesini gerektirir; sıfır gecikmeli veya gerçek zamanlı çalışma ölçülmüş değildir.

In [ ]:
def enhance(noisy,p,mean,std):
    spectrum=transform.stft(noisy)
    X=np.log1p(abs(spectrum)).T.astype(np.float32)[None,:,:]
    masks=forward_gru((X-mean)/std,p,keep_cache=False)[0][0]
    output=transform.istft(masks.T*spectrum,k1=len(noisy))
    assert np.isfinite(output).all() and masks.min()>=0 and masks.max()<=1
    return output


def score(split,p,mean,std):
    results=[]
    for row in pairs:
        if row['split']!=split:continue
        clean=read_audio(row['clean_path']);noisy=read_audio(row['noisy_path'])
        output=enhance(noisy,p,mean,std)
        before=snr(clean,noisy);after=snr(clean,output)
        results.append({'clean_id':row['clean_id'],'snr':int(row['target_snr_db']),
                        'before':before,'after':after,'improvement':after-before})
    for level in [0,5,10]:
        rows=[r for r in results if r['snr']==level]
        print(split,level,{k:float(np.mean([r[k] for r in rows])) for k in ['before','after','improvement']},flush=True)
    return results


validation_results = score('validation', gru_params, gru_mean, gru_std)
test_results = score('test', gru_params, gru_mean, gru_std)
for split, rows in [('validation', validation_results), ('test', test_results)]:
    with (OUTPUT_DIR / f'{split}_snr.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader(); writer.writerows(rows)

## 7. Sabit örneği dinleme

Önceki modelle aynı clean_000, 5 dB test kaydı seçilir. En iyi sonucu veren ses
sonradan aranmaz. Üç kayda ortak oynatma kazancı uygulanır. Paketteki 4 WAV ise
temiz, gürültülü, önceki maske MLP'si ve GRU çıkışlarını aynı kazançla karşılaştırır.

In [ ]:
from IPython.display import Audio, display

# Önceki sürümlerle aynı örnek; en iyi duyulan kaydı sonradan aramıyoruz.
row = next(r for r in pairs if r['split'] == 'test' and r['target_snr_db'] == '5')
reference = read_audio(row['clean_path'])
noisy = read_audio(row['noisy_path'])
gru_output = enhance(noisy, gru_params, gru_mean, gru_std)
print('Örnek:', row['clean_id'])
print('Giriş SNR:', snr(reference, noisy))
print('GRU çıkış SNR:', snr(reference, gru_output))
signals = [('clean_reference', reference), ('noisy_input', noisy), ('gru_output', gru_output)]
gain = 0.95 / max(1.0, *[float(np.max(np.abs(x))) for _, x in signals])
for label, signal in signals:
    print(label)
    playback = signal * gain
    display(Audio(playback, rate=16000, normalize=False))
    wavfile.write(OUTPUT_DIR / f'{label}.wav', 16000,
                 np.rint(playback * 32767).astype(np.int16))

## 8. Sonuçları indirme

Test grubu önceki geliştirme denemelerinde görülmüştür. Bu sonuçları küçük DNS alt
kümesinde geliştirme karşılaştırması olarak sunun; yeni, kör benchmark olarak değil.
Nihai genelleme için daha önce kullanılmamış konuşmacı ve gürültü kaynakları gerekir.

In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/GRU_Sonuclarim', 'zip', OUTPUT_DIR)
files.download(archive)

## Kaynaklar ve derslerle ilişkisi

- Course 1: ileri yayılım, maliyet, zincir kuralı ve gradyanla öğrenme.
- Course 2: normalizasyon, doğrulama ayrımı, Adam ve model seçimi.
- Course 5: diziler, recurrent gizli durum, kapılar ve zaman boyunca geri yayılım.

Bu eğitim deneyi, derslerde öğrenilen temel fikirlerin elle yazılmış NumPy uygulamasıdır;
derslerin belirli bir ödevinin aynısı veya nihai SNN projesi değildir.

- Cho et al. (2014), gated recurrent unit fikri ve RNN:
  https://arxiv.org/html/1406.1078v3#S2.SS3
- Goodfellow et al., Deep Learning, Chapter 10:
  https://www.deeplearningbook.org/contents/rnn.html
- Phase-sensitive speech enhancement hedefi: Erdoğan et al., ICASSP 2015:
  https://research.sabanciuniv.edu/id/eprint/28858/
- DNS3 veri kaynakları ve lisanslar: önceki paketin sources.csv dosyası;
  https://github.com/microsoft/DNS-Challenge/blob/master/README-DNS3.md